# **Real-Time Object Detection with YOLO**

Welcome to Lab #3. In this lab you will build a complete object-detection pipeline. You will create and annotate your own dataset for one object class, train a *YOLO11 Nano* model on a Colab GPU, and run it as a real-time webcam application on your laptop.

You will learn to:
- collect images and annotate them with bounding boxes in the YOLO label format
- prepare a train/validation split and a dataset configuration file
- fine-tune a pretrained YOLO11n detector on your own data (transfer learning)
- read training curves and detection metrics (precision, recall, mAP)
- deploy the trained model in a real-time webcam application with Streamlit

# Lab overview

The lab has three parts. Only Part 2 runs in this notebook.

| Part | Where | What you do |
|---|---|---|
| **Part 1:** Data collection and annotation | Your laptop | Photograph one object, draw bounding boxes in LabelImg, zip the folder |
| **Part 2:** Training (Tasks 1–9) | This notebook, on Google Colab with a GPU (or on your own computer) | Check and split the data, train YOLO11n, evaluate it, prepare the app and download your files |
| **Part 3:** Real-time deployment | Your laptop | Run the webcam app with your trained model and test it |

**What you need:** a laptop with a webcam, Python 3.10–3.12 installed on it, a Google account (for Colab), and a phone or camera to take photos.

# Important

1- There are **9 tasks** in this notebook (Part 2). Parts 1 and 3 are step-by-step instructions that you carry out on your laptop.

2- Write all answers in the accompanying **Lab 3 Report Template**, not in this notebook. Cells marked **Report:** tell you which section each result belongs to: dataset details and evidence (Section 1), training settings (Section 2), training results (Section 3), the webcam tests (Section 4), questions Q1–Q4 (Section 5) and your reflection (Section 6).

3- **GPU:** in Colab, choose **Runtime → Change runtime type → T4 GPU** before you start. You can also run this notebook on your own computer; see *Running on your own computer* at the start of Part 2.

4- Free Colab sessions disconnect when idle, and everything in `/content` is then deleted. Keep `dataset.zip` on your computer, and download your results at the end of Part 2 before you close the notebook.

5- **Start Part 1 early.** Taking and annotating about 100 photos takes one to two hours.

# Part 1: Data collection and annotation (on your laptop)

Choose **one object class** that you can bring to the lab demonstration, for example a coffee mug, a computer mouse, sunglasses, a watch or a water bottle. Pick an object that is not tiny, so it is easy to see and to annotate. Do not choose people or faces.

## Step 1.1: Collect images

1. Take **100–120 photos** of your object. A phone camera is fine.
2. Vary the conditions, so the model learns the object and not one particular scene:
   - **Angles:** top-down, side view, front view.
   - **Lighting:** bright light, dim light, shadows.
   - **Background:** clean desk, cluttered desk or floor, held in your hand (as in the webcam demo).
   - **Distance:** close-up and further away.
3. The object should be visible in every photo. If it appears more than once in a photo, you will label every copy.
4. Practical tips:
   - Hold the phone **horizontally (landscape)**. This matches the webcam view and avoids rotated-image problems in LabelImg.
   - Use **JPG or PNG**. iPhones save HEIC by default, which YOLO cannot read: choose *Settings → Camera → Formats → Most Compatible* before taking the photos.
   - Very large photos make the upload to Colab slow. A medium resolution is enough, because the images are resized to 640 pixels for training.
5. On your Desktop, create a folder named `my_dataset`, and inside it a subfolder named `images`. Put all your photos in `my_dataset/images`.

**Report:** Take a screenshot of the `images` folder showing your photos for **Figure 1** in **Section 1.2** of the Lab 3 Report Template.

## Step 1.2: Install LabelImg

We use LabelImg, a free tool for drawing bounding boxes. Open a Command Prompt (Windows) or Terminal (Mac/Linux) and run:

```
pip install labelImg
labelImg
```

**If LabelImg does not work:** LabelImg is no longer maintained, and with newer Python versions it can crash (for example with `TypeError: ... unexpected type 'float'` when you draw a box or zoom). In that case, use the free browser tool [makesense.ai](https://www.makesense.ai) instead, as described at the end of Step 1.3. Nothing needs to be installed for it.

## Step 1.3: Annotate the images

1. **Set the format to YOLO.** In the toolbar on the left, click the button labelled **PascalVOC** until it shows **YOLO**. If you skip this, your labels will be in the wrong format and training will fail.
2. Click **Open Dir** and choose `my_dataset/images`. Then click **Change Save Dir** and choose the **same** folder.
3. Turn on **View → Auto Save mode**, so each image is saved when you move to the next one.
4. For each image:
   - press **W** and draw a **tight** box around the whole visible object;
   - type your label name in lower case (for example `mug`) and click OK. Use **exactly the same name** every time;
   - press **D** to go to the next image (**A** goes back).
5. Each image now has a `.txt` file with the same name next to it, and LabelImg also creates a `classes.txt` file. Keep both.

Each line of a label file describes one box, for example `0 0.512 0.430 0.250 0.310`. You will look at what these numbers mean in Task 3. Don't worry if the first number (the class id) is not 0: LabelImg numbers classes by their position in its own list, and this notebook fixes the numbering for you.

**If you use makesense.ai instead of LabelImg:**
1. Open [makesense.ai](https://www.makesense.ai), click *Get Started*, drop in all the photos from `my_dataset/images` and choose *Object Detection*.
2. Create **one** label with the name of your object (for example `mug`) and start the project.
3. In each image, draw a tight rectangle around the object and select your label for it.
4. Choose *Actions → Export Annotations → "A .zip package containing files in YOLO format"*. Unzip the downloaded file and copy all the `.txt` files into `my_dataset/images`, next to the photos. (makesense.ai does not create `classes.txt`; that is fine.)
5. Your browser shows portrait phone photos upright, so your boxes match the upright photo. If the boxes look rotated in the check after Task 3, set `APPLY_EXIF_ROTATION = True` there.

**Report:** Take a screenshot of LabelImg (or makesense.ai) showing one of your images with its bounding box for **Figure 2** in **Section 1.2**.

## Step 1.4: Zip the dataset

Do **not** split the images into training and validation sets yourself: the notebook does this in Task 4.

Compress the whole `my_dataset` folder into a file named **`dataset.zip`**:
- **Windows:** right-click `my_dataset` → *Compress to ZIP file* (or *Send to → Compressed (zipped) folder*), then rename the file to `dataset.zip`.
- **Mac:** right-click `my_dataset` → *Compress*, then rename the file to `dataset.zip`.

The zip file should look like this:

```
dataset.zip
└── my_dataset/
    └── images/
        ├── IMG_0001.jpg
        ├── IMG_0001.txt
        ├── IMG_0002.jpg
        ├── IMG_0002.txt
        ├── ...
        └── classes.txt
```

# Part 2: Training (this notebook)

We recommend **Google Colab**, which gives you a free GPU. You can also run this notebook in Jupyter on your own computer: the code detects where it is running and adapts the file locations.

**Running on your own computer (optional)**
1. Create the project folder and virtual environment from **Step 3.1** in Part 3 (points 1–4) now, and keep the environment activated. You will use the same environment for the webcam app.
2. If your computer has an NVIDIA GPU, first install PyTorch with GPU support: on [pytorch.org](https://pytorch.org/get-started/locally/) choose your system with *Pip* and a *CUDA* version, and run the command it shows. It installs `torch` and `torchvision` together, and the two must come from the same command.
3. Install the other packages:
   ```
   pip install ultralytics streamlit jupyter pyyaml
   ```
4. Put this notebook and `dataset.zip` in the project folder, run `jupyter notebook` in that folder and open the notebook.
5. Everything the notebook creates is saved in that folder. Without a GPU, training takes much longer (an hour or more for 50 epochs); you can reduce `NUM_EPOCHS` in Task 1.

## 2.1 Setup

The next cell installs the [Ultralytics](https://docs.ultralytics.com) package, which provides the YOLO11 models, if it is not installed yet. The cell after it imports the packages and checks for a GPU. In Colab it should report a **Tesla T4**; if it reports the CPU, choose **Runtime → Change runtime type → T4 GPU** and run the cells again.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Google Colab' if IN_COLAB else 'Running locally')

try:
    import ultralytics
except ImportError:
    %pip install -q ultralytics

In [ ]:
import random
import shutil
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
import yaml
import ultralytics
from ultralytics import YOLO

ultralytics.checks()

# Use the GPU if PyTorch and torchvision can both use it, otherwise the CPU
DEVICE = 'cpu'
if torch.cuda.is_available():
    import torchvision
    try:
        torchvision.ops.nms(torch.zeros(1, 4, device='cuda'), torch.ones(1, device='cuda'), 0.5)
        DEVICE = 0
    except Exception:
        print('WARNING: PyTorch can use your GPU, but your torchvision installation cannot, so training will run on the CPU.\n'
              'To use the GPU, reinstall torch and torchvision together with the command from pytorch.org.')
print('Training device:', f'GPU ({torch.cuda.get_device_name(0)})' if DEVICE == 0 else 'CPU')

WORK_DIR = Path('/content') if IN_COLAB else Path.cwd()   # all files are created in this folder
print('Working folder:', WORK_DIR)

## 2.2 Upload your dataset

**In Colab:** click the **folder icon** in the left sidebar and drag `dataset.zip` into the file list. Wait until the upload has finished (the progress circle at the bottom of the sidebar disappears). If you prefer, simply run the next cell: it opens a file dialog when `dataset.zip` is not found.

**On your own computer:** put `dataset.zip` in the same folder as this notebook.

In [ ]:
ZIP_PATH = WORK_DIR / 'dataset.zip'

if not ZIP_PATH.exists():
    if not IN_COLAB:
        raise FileNotFoundError(f'Put dataset.zip in {WORK_DIR} (the folder of this notebook) and run this cell again.')
    from google.colab import files
    print('dataset.zip was not found in /content. Choose it in the dialog below.')
    uploaded = files.upload()
    # Colab renames repeated uploads (e.g. "dataset (1).zip"), so rename the file back
    Path('/content', next(iter(uploaded))).rename(ZIP_PATH)

print(f'{ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)')

Unzip the dataset and list the images and the class names that LabelImg saved.

In [ ]:
RAW_DIR = WORK_DIR / 'lab3_raw'
shutil.rmtree(RAW_DIR, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW_DIR)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_files = sorted(p for p in RAW_DIR.rglob('*')
                     if p.suffix.lower() in IMG_EXTS and '__MACOSX' not in p.parts)
print('Images found:', len(image_files))

heic = [p for p in RAW_DIR.rglob('*') if p.suffix.lower() in {'.heic', '.heif'}]
if heic:
    print(f'WARNING: {len(heic)} HEIC images were found. YOLO cannot read them: convert them to JPG and zip again.')

classes_files = [p for p in RAW_DIR.rglob('classes.txt') if '__MACOSX' not in p.parts]
label_names = classes_files[0].read_text().split() if classes_files else []
print('Class names in classes.txt:', label_names if label_names else '(no classes.txt found; that is fine if you used makesense.ai)')

### Task 1: Settings

Set the name of your object class (the label you typed in LabelImg), the number of training epochs and the training image size.

In [ ]:
# Task 1
### START CODE HERE ###
CLASS_NAME =     # The label name you used in LabelImg, as a string, e.g. 'mug'
NUM_EPOCHS =     # Number of training epochs (50 is a good starting point)
IMG_SIZE =       # Training image size in pixels (use 640)
### END CODE HERE ###

BATCH_SIZE = 16  # Images per training batch (16 works well on a T4 GPU)
SEED = 0         # Fixes the random train/validation split and training, so results can be reproduced

### Task 2: Check your annotations

Each image needs a label file with the **same name** and the extension `.txt`, in the same folder. For example, the labels for `my_dataset/images/IMG_0001.jpg` are in `my_dataset/images/IMG_0001.txt`.

Complete `label_path_for()` so that it returns the label file path for an image path. Then run the cell below it, which checks every label file and counts the images and boxes.

In [ ]:
# Task 2
def label_path_for(image_path):
    """Return the path of the YOLO label file that belongs to image_path (a pathlib.Path)."""
    ### START CODE HERE ### (≈ 1 line)
    # Hint: Path.with_suffix() replaces the file extension
    return
    ### END CODE HERE ###

In [ ]:
def parse_label_file(label_path):
    """Read a YOLO label file. Returns (boxes, n_bad): boxes is a list of
    (class_id, x_center, y_center, width, height); n_bad counts invalid lines."""
    boxes, n_bad = [], 0
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        try:
            cid = int(float(parts[0]))
            xc, yc, w, h = (float(v) for v in parts[1:])   # fails unless there are exactly 4 numbers
            if not all(0 <= v <= 1 for v in (xc, yc, w, h)):
                raise ValueError
            boxes.append((cid, xc, yc, w, h))
        except ValueError:
            n_bad += 1
    return boxes, n_bad


labelled_images, missing, background = [], [], []
n_boxes, n_bad, class_counts = 0, 0, {}
for img in image_files:
    lbl = label_path_for(img)
    if not lbl.exists():
        missing.append(img.name)
        continue
    boxes, bad = parse_label_file(lbl)
    labelled_images.append(img)
    n_boxes += len(boxes)
    n_bad += bad
    if not boxes:
        background.append(img.name)
    for cid, *_ in boxes:
        class_counts[cid] = class_counts.get(cid, 0) + 1

print(f'Images found:                {len(image_files)}')
print(f'Images with a label file:    {len(labelled_images)}')
print(f'Bounding boxes:              {n_boxes}')
if missing:
    print(f'WARNING: {len(missing)} images have no label file and will not be used, e.g. {missing[:3]}')
if background:
    print(f'Note: {len(background)} label files are empty (YOLO treats these images as background), e.g. {background[:3]}')
if n_bad:
    print(f'WARNING: {n_bad} label lines are not in YOLO format and will be ignored. Did you set LabelImg to YOLO?')

names_used = {cid: (label_names[cid] if cid < len(label_names) else '?') for cid in sorted(class_counts)}
if label_names:
    print('Boxes per class id:', class_counts, '-> names in classes.txt:', names_used)
else:
    print('Boxes per class id:', class_counts)
if len(class_counts) > 1:
    print('WARNING: your labels use more than one class id. This lab uses one class, so all boxes will be '
          f'treated as "{CLASS_NAME}". If you labelled two different objects, fix the labels and zip again.')
if label_names and CLASS_NAME not in names_used.values():
    print(f'WARNING: CLASS_NAME = "{CLASS_NAME}" does not match the label names used in your files {list(names_used.values())}.')

**Report:** Enter the number of photos you collected, the number of images with a label file and the number of bounding boxes in **Section 1.1** of the Lab 3 Report Template.

### Task 3: The YOLO label format

Each line of a YOLO label file describes one bounding box with five numbers:

```
class_id  x_center  y_center  width  height
```

The four box values are **normalised** by the image size: `x_center` and `width` are divided by the image width, and `y_center` and `height` by the image height, so they lie between 0 and 1. The point (0, 0) is the **top-left** corner of the image.

To draw a box, we need its corners in pixels: the top-left corner `(x1, y1)` and the bottom-right corner `(x2, y2)`. Note that the label gives the **centre** of the box, not a corner.

**Example.** An image is 1000 × 800 pixels and a label line is `0 0.40 0.30 0.20 0.25`.
- Centre in pixels: x = 0.40 × 1000 = 400, y = 0.30 × 800 = 240
- Box size in pixels: width = 0.20 × 1000 = 200, height = 0.25 × 800 = 200
- The corners are half a box away from the centre:
  - x1 = 400 − 200/2 = 300, y1 = 240 − 200/2 = 140
  - x2 = 400 + 200/2 = 500, y2 = 240 + 200/2 = 340

```
(x1, y1) ┌──────────────┐
         │              │
         │   ● centre   │ height
         │              │
         └──────────────┘ (x2, y2)
              width
```

Complete `yolo_to_corners()` to compute the four corner values for any box and image size.

In [ ]:
# Task 3
def yolo_to_corners(xc, yc, w, h, img_w, img_h):
    """Convert one normalised YOLO box to pixel corners (x1, y1, x2, y2)."""
    ### START CODE HERE ### (≈ 4 lines)
    x1 =
    y1 =
    x2 =
    y2 =
    ### END CODE HERE ###
    return round(x1), round(y1), round(x2), round(y2)

print(yolo_to_corners(0.5, 0.5, 0.25, 0.5, img_w=640, img_h=480))

**Expected output:** `(240, 120, 400, 360)`

Now draw your own labels on six random images. **Check that every green box fits tightly around your object.**

If the boxes are clearly in the wrong place or rotated by 90° relative to the object, your photos were stored with a rotation flag that your annotation tool and OpenCV read differently. Set `APPLY_EXIF_ROTATION = True` in the next cell and run it again. Use the setting for which the boxes fit: the split in Task 4 saves the images exactly as they are shown here.

In [ ]:
APPLY_EXIF_ROTATION = False   # LabelImg ignores the phone's rotation flag; set True if the boxes look rotated

def load_image(path):
    """Read an image as a BGR NumPy array, with or without the phone's rotation flag (EXIF) applied."""
    flags = cv2.IMREAD_COLOR if APPLY_EXIF_ROTATION else cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION
    return cv2.imread(str(path), flags)

def draw_labels(image_path):
    img = load_image(image_path)
    img_h, img_w = img.shape[:2]
    boxes, _ = parse_label_file(label_path_for(image_path))
    for cid, xc, yc, w, h in boxes:
        x1, y1, x2, y2 = yolo_to_corners(xc, yc, w, h, img_w, img_h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), max(2, img_w // 150))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

random.seed(SEED)
examples = random.sample(labelled_images, min(6, len(labelled_images)))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax in axes.flat:
    ax.axis('off')
for ax, path in zip(axes.flat, examples):
    ax.imshow(draw_labels(path))
    ax.set_title(path.name if len(path.name) <= 30 else '…' + path.name[-29:], fontsize=10)
plt.tight_layout()
plt.show()

**Report:** Open one of your own label files, copy one line into **Section 1.3** and convert it to pixel corners, as asked there.

### Task 4: Split the data into training and validation sets

YOLO trains on the **training set** and measures accuracy on the **validation set**, which it never trains on. Use **80%** of the images for training and **20%** for validation.

The list `images` contains the paths of all your labelled images. Complete the code in three steps:

1. **Shuffle** the list with `random.shuffle(images)`. This mixes the list in place, so the photos you took one after another do not all end up in the same set. (It returns nothing, so do not write `images = random.shuffle(images)`.)
2. **Count** the training images: 80% of `len(images)`, rounded down to a whole number with `int()`. For example, 120 images give `int(0.8 * 120) = 96`.
3. **Split** the list with slicing: `images[:n_train]` gives the first `n_train` items, and `images[n_train:]` gives all the items after them.

**Example** with a list of five letters:

```python
letters = ['a', 'b', 'c', 'd', 'e']
random.shuffle(letters)         # e.g. ['d', 'a', 'e', 'b', 'c']
n = int(0.8 * len(letters))     # int(4.0) = 4
letters[:n]                     # ['d', 'a', 'e', 'b']  (the first 4)
letters[n:]                     # ['c']                 (the rest)
```

**Expected output** for 120 labelled images: `Training images: 96` and `Validation images: 24`.

The cell after the task copies the images and labels into the folder structure that YOLO expects:

```
lab3_yolo_dataset/
├── train/images   train/labels
└── val/images     val/labels
```

While copying, large photos are scaled down to at most 1280 pixels (this makes training faster; the normalised labels stay correct), and every box is given class id 0, the id of the only class in a one-class dataset.

In [ ]:
# Task 4
random.seed(SEED)
images = list(labelled_images)
### START CODE HERE ### (≈ 4 lines)
# Step 1: shuffle the list `images` (put the list inside the brackets)
random.shuffle(  )
# Step 2: number of training images = 80% of len(images), as a whole number (use int())
n_train =
# Step 3: the first n_train images for training, the remaining images for validation (use slicing)
train_images =
val_images =
### END CODE HERE ###

print('Training images:  ', len(train_images))
print('Validation images:', len(val_images))

In [ ]:
DATASET_DIR = WORK_DIR / 'lab3_yolo_dataset'
MAX_SIDE = 1280
shutil.rmtree(DATASET_DIR, ignore_errors=True)

def copy_split(image_list, split):
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    img_dir.mkdir(parents=True)
    lbl_dir.mkdir(parents=True)
    used = set()
    for src in image_list:
        stem = src.stem
        while stem in used:            # avoid name clashes such as IMG_1.jpg and IMG_1.png
            stem += '_1'
        used.add(stem)
        img = load_image(src)
        h, w = img.shape[:2]
        scale = MAX_SIDE / max(h, w)
        if scale < 1:
            img = cv2.resize(img, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(img_dir / f'{stem}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        boxes, _ = parse_label_file(label_path_for(src))
        lines = [f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}' for _, xc, yc, bw, bh in boxes]
        (lbl_dir / f'{stem}.txt').write_text(''.join(line + '\n' for line in lines))

copy_split(train_images, 'train')
copy_split(val_images, 'val')
for split in ['train', 'val']:
    print(split, len(list((DATASET_DIR / split / 'images').iterdir())), 'images,',
          len(list((DATASET_DIR / split / 'labels').iterdir())), 'label files')

**Report:** Enter the numbers of training and validation images in **Section 1.1**.

### Task 5: Dataset configuration file

YOLO reads where your data is, and what your class is called, from a small YAML file. In Python you first fill in a **dictionary** with five entries; the code below then saves it as `data.yaml`.

| Key | What to enter | Type |
|---|---|---|
| `'path'` | the folder that contains `train` and `val`: the variable `DATASET_DIR` from Task 4, converted to text with `str()` | text |
| `'train'` | the folder with the training **images**, relative to `path` | text in quotes, e.g. `'folder/subfolder'` |
| `'val'` | the folder with the validation **images**, relative to `path` | text in quotes |
| `'nc'` | the number of classes | whole number |
| `'names'` | a list with your class name: the variable `CLASS_NAME` from Task 1, inside square brackets `[ ]` | list |

Look at the folder tree in Task 4 to find the `train` and `val` image folders. You do not need to give the label folders: YOLO finds them by replacing `images` with `labels` in the path.

**Expected output** in Colab for `CLASS_NAME = 'mug'` (on your own computer, the `path` line shows your own folder):

```
path: /content/lab3_yolo_dataset
train: train/images
val: val/images
nc: 1
names:
- mug
```

In [ ]:
# Task 5
DATA_YAML = WORK_DIR / 'data.yaml'
### START CODE HERE ### (≈ 5 lines)
data_config = {
    'path':    ,   # the dataset folder from Task 4, as text: str(...)
    'train':   ,   # the training images folder, relative to 'path' (text in quotes)
    'val':     ,   # the validation images folder, relative to 'path' (text in quotes)
    'nc':      ,   # the number of classes (a whole number)
    'names':   ,   # a list with your class name: CLASS_NAME inside [ ]
}
### END CODE HERE ###

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)
print(DATA_YAML.read_text())

## 2.3 YOLO11 Nano

YOLO ("You Only Look Once") is a **single-stage** detector: one forward pass of the network predicts all bounding boxes and their class scores at once, which makes it fast enough for real-time video. YOLO11 predicts boxes at three scales, so it can find both small and large objects, and removes duplicate boxes for the same object with non-maximum suppression (NMS).

YOLO11 comes in five sizes, from **n** (nano) to **x** (extra large). The larger models are more accurate but slower; see the model table in the [Ultralytics YOLO11 documentation](https://docs.ultralytics.com/models/yolo11/).

The checkpoint `yolo11n.pt` was trained on the COCO dataset (80 everyday classes). We **fine-tune** it on your images: YOLO replaces the 80-class output layer with a 1-class one and reuses all other learned weights. This is **transfer learning**, and it is why about 100 images are enough.

### Task 6: Train YOLO11n

Load the pretrained model and train it on your dataset. Training prints one progress line per epoch. With about 100 images and 50 epochs it takes roughly 5–15 minutes on a T4 GPU (much longer on a CPU).

In [ ]:
# Task 6
RUNS_DIR = WORK_DIR / 'lab3_runs'
### START CODE HERE ### (≈ 2 lines)
# 1. Load the pretrained YOLO11 Nano model from the checkpoint 'yolo11n.pt' with YOLO()
model =
# 2. Train it with model.train(), passing: data=str(DATA_YAML), epochs=NUM_EPOCHS, imgsz=IMG_SIZE,
#    batch=BATCH_SIZE, device=DEVICE, seed=SEED, plots=True, project=str(RUNS_DIR), name='train', exist_ok=True
results =
### END CODE HERE ###

save_dir = Path(model.trainer.save_dir)
print('Training results are saved in', save_dir)

**Report:** Complete the training settings in **Section 2**. The device (GPU name, or CPU) is shown in the first lines of the training output, and the training time in the line *"… epochs completed in … hours"* near the end.

### Training curves

During training, YOLO saves the losses and metrics of every epoch in `results.csv` and plots them in `results.png`. YOLO uses three losses: the **box loss** (how well the predicted boxes match the labelled boxes), the **classification loss** (how confident and correct the class scores are) and the **DFL loss** (a refinement of the box edges).

In `results.png`, the **top row** shows the three **training** losses, followed by precision and recall. The **bottom row** shows the same three losses on the **validation** set, followed by mAP50 and mAP50-95. To compare training and validation, look at each training loss and the validation loss directly below it.

*Note:* in the last 10 epochs YOLO turns off its "mosaic" data augmentation (which combines four images into one), so the training losses often drop suddenly there. This is expected.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(save_dir / 'results.png'), width=1000))

**Report:** Insert `results.png` as **Figure 3** in **Section 3.1**. You will use it to answer Q2.

### Task 7: Evaluate the best model

YOLO saves two checkpoints: `last.pt` (after the final epoch) and `best.pt` (the epoch with the best validation score). We evaluate `best.pt` on the validation set. The metrics are:

- **IoU (Intersection over Union):** the overlap of a predicted box and a labelled box, divided by the area of their union. 1 means a perfect match.
- **Precision:** the fraction of predicted boxes that are correct (few false positives).
- **Recall:** the fraction of labelled objects that were found (few false negatives).
- **mAP50:** the mean average precision when a prediction counts as correct if its IoU with a labelled box is at least 0.5. It summarises precision and recall over all confidence thresholds.
- **mAP50-95:** the mAP averaged over the IoU thresholds 0.50, 0.55, …, 0.95, so it also rewards boxes that are placed very precisely.

Complete the code to load `best.pt` and evaluate it.

In [ ]:
# Task 7
best_path = save_dir / 'weights' / 'best.pt'
### START CODE HERE ### (≈ 2 lines)
# 1. Load your trained model from best_path with YOLO()
best_model =
# 2. Evaluate it on the validation set with best_model.val(), passing data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE
metrics =
### END CODE HERE ###

print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall:    {metrics.box.mr:.3f}')
print(f'mAP50:     {metrics.box.map50:.3f}')
print(f'mAP50-95:  {metrics.box.map:.3f}')
print('Time per image (ms):', {k: round(v, 1) for k, v in metrics.speed.items()})

The confusion matrix from training shows how many of your objects were found (true positives), missed (counted as "background") or wrongly detected in the background (false positives).

In [ ]:
display(Image(filename=str(save_dir / 'confusion_matrix.png'), width=600))

**Report:** Enter precision, recall, mAP50, mAP50-95 and the inference time per image in **Section 3.2**.

### Task 8: Detections on validation images

Run your trained model on eight validation images and look at the results. Each detection is drawn with its class name and confidence score. Only boxes with a confidence of at least `conf` are kept.

In [ ]:
# Task 8
val_dir = DATASET_DIR / 'val' / 'images'
sample = [str(p) for p in sorted(val_dir.iterdir())[:8]]
### START CODE HERE ### (≈ 1 line)
# Run best_model.predict() on the list `sample`, with conf=0.25, device=DEVICE and verbose=False
preds =
### END CODE HERE ###

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax in axes.flat:
    ax.axis('off')
for ax, r in zip(axes.flat, preds):
    ax.imshow(r.plot()[:, :, ::-1])   # plot() returns a BGR image; [:, :, ::-1] converts it to RGB
    name = Path(r.path).name
    ax.set_title(f'{name if len(name) <= 30 else "…" + name[-29:]}\n{len(r.boxes)} detection(s)', fontsize=10)
plt.tight_layout()
fig.savefig(WORK_DIR / 'predictions.png', dpi=120)
plt.show()

**Report:** Insert the figure (`predictions.png`) as **Figure 4** in **Section 3.3** and point out any missed objects or false detections.

### Task 9: The webcam application

In Part 3 you will run your model on your laptop's webcam with a small [Streamlit](https://streamlit.io) web app. The app reads one frame from the webcam, runs the model on it, draws the detections and shows the frame in the browser, over and over in a loop. Above the video it shows the number of detections, the model's inference time in milliseconds and the number of frames per second (FPS) the loop achieves.

Complete the three missing lines inside the loop of `app.py` below. They do the same as Task 8, but for one webcam frame at a time:

1. **Run the model on the frame.** Calling the model like a function runs detection on an image: `model(image)`. Also pass the slider value as the confidence threshold with `conf=conf`, and `verbose=False` so that it does not print a line for every frame. The result is a list with one entry per image; since you give it one frame, its result is `results[0]`.
2. **Draw the boxes.** `results[0].plot()` returns a copy of the frame with the boxes, class name and confidence scores drawn on it (you used `r.plot()` the same way in Task 8).
3. **Fix the colours.** OpenCV stores images in **BGR** colour order, but the browser expects **RGB**. Convert the image with `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)`, using `annotated` as the image.

Then run the cell. `%%writefile` saves the code as the file `app.py` in the working folder; it does not start the app, because the app needs a webcam, which Colab does not have.

The cell after it checks `app.py` for syntax errors and then **tests your three lines on one validation image**. You should see the image with your detections in natural colours. If the colours look wrong (for example, a blue tint or blue skin), step 3 is missing or wrong.

In [ ]:
%%writefile app.py
import time

import cv2
import streamlit as st
from ultralytics import YOLO

st.title("Real-Time Object Detection")
st.write("Tick **Run camera** to start the webcam, and untick it to stop.")


@st.cache_resource
def load_model(path):
    return YOLO(path)


model = load_model("best.pt")      # your trained model, in the same folder as app.py
conf = st.sidebar.slider("Confidence threshold", 0.05, 0.95, 0.25, 0.05)
camera_id = st.sidebar.number_input("Camera ID", min_value=0, max_value=5, value=0, step=1)
run = st.checkbox("Run camera")

info = st.empty()
frame_placeholder = st.empty()

if run:
    cap = cv2.VideoCapture(int(camera_id))
    if not cap.isOpened():
        st.error("Could not open the webcam. Close other apps that use it, or try another Camera ID.")
    try:
        prev = time.perf_counter()
        while cap.isOpened():
            ok, frame = cap.read()             # frame is a BGR image (NumPy array)
            if not ok:
                st.warning("Could not read a frame from the webcam.")
                break

            ### START CODE HERE ### (≈ 3 lines)
            # Step 1: run the model on `frame`, with conf=conf and verbose=False
            results =
            # Step 2: draw the boxes: call .plot() on the first result, results[0]
            annotated =
            # Step 3: convert `annotated` from BGR to RGB with cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            frame_rgb =
            ### END CODE HERE ###

            frame_placeholder.image(frame_rgb, channels="RGB")
            now = time.perf_counter()
            fps = 1.0 / (now - prev)
            prev = now
            info.write(f"Detections: {len(results[0].boxes)} | "
                       f"Inference: {results[0].speed['inference']:.0f} ms | Loop: {fps:.1f} FPS")
    finally:
        cap.release()                          # free the webcam when the loop stops

In [ ]:
import ast, textwrap
source = Path('app.py').read_text(encoding='utf-8')
try:
    ast.parse(source)
    print('app.py has no syntax errors ✓')
except SyntaxError as e:
    print(f'Syntax error in app.py, line {e.lineno}: {e.text}')
else:
    # Test your three lines from app.py on one validation image instead of a webcam frame
    block = source.split('### START CODE HERE ###')[1].split('### END CODE HERE ###')[0]
    block = textwrap.dedent(block.split('\n', 1)[1])
    test = {'model': best_model, 'frame': cv2.imread(sample[0]), 'conf': 0.25, 'cv2': cv2}
    exec(block, test)
    plt.figure(figsize=(6, 5))
    plt.imshow(test['frame_rgb'])
    plt.title(f"Test of your app code: {len(test['results'][0].boxes)} detection(s)")
    plt.axis('off')
    plt.show()

## 2.4 Download your files

The next cell collects the files you need for Part 3 and for the report into `lab3_results.zip`. In Colab it then downloads the file; on your own computer the file is saved in the notebook's folder.
- `best.pt` (your trained model) and `app.py`, for Part 3
- `results.png`, `confusion_matrix.png` and `predictions.png`, for the report

In Colab, if the download does not start, open the folder icon in the sidebar, right-click `lab3_results.zip` and choose *Download*.

In [ ]:
results_zip = WORK_DIR / 'lab3_results.zip'
to_save = {
    'best.pt': best_path,
    'app.py': WORK_DIR / 'app.py',
    'results.png': save_dir / 'results.png',
    'confusion_matrix.png': save_dir / 'confusion_matrix.png',
    'predictions.png': WORK_DIR / 'predictions.png',
}
with zipfile.ZipFile(results_zip, 'w') as z:
    for name, path in to_save.items():
        if path.exists():
            z.write(path, name)
        else:
            print('Missing (run the task that creates it):', name)
print(f'{results_zip} ({results_zip.stat().st_size / 1e6:.1f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(str(results_zip))

**Before you close the notebook:** save it with its outputs (in Colab: *File → Download → Download .ipynb*). You will submit it with your report.

# Part 3: Real-time deployment (on your laptop)

## Step 3.1: Create the project folder and environment

*If you ran Part 2 on your own computer, you already have the folder and the environment: unzip `lab3_results.zip` in that folder and go to Step 3.2.*

1. Unzip `lab3_results.zip` into a new folder on your Desktop named `yolo_project`. It must contain `best.pt` and `app.py`.
2. Open a Command Prompt (Windows) or Terminal (Mac/Linux) and go to the folder:
   ```
   cd Desktop/yolo_project
   ```
3. Create a virtual environment, so the packages for this lab are kept separate from the rest of your system:
   ```
   python -m venv venv
   ```
4. Activate it. You should then see `(venv)` at the start of the prompt.
   - Windows: `venv\Scripts\activate`
   - Mac/Linux: `source venv/bin/activate`

   *In Windows PowerShell, if activation is blocked, run `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` once, or use the Command Prompt instead.*
5. Install the packages (this downloads PyTorch and can take a few minutes):
   ```
   pip install ultralytics streamlit
   ```

## Step 3.2: Run the app

With the virtual environment activated, run:

```
streamlit run app.py
```

A browser tab opens. Tick **Run camera** and hold your object in front of the webcam. Use the slider in the sidebar to change the confidence threshold. To stop the app, untick **Run camera** and press **Ctrl+C** in the terminal.

**Troubleshooting**
- *"Could not open the webcam":* close other apps that use the camera (Teams, Zoom, …), or set **Camera ID** to 1. On a Mac, allow your terminal to use the camera (*System Settings → Privacy & Security → Camera*).
- *`best.pt` not found:* start Streamlit from the folder that contains both `app.py` and `best.pt`.
- *An error mentioning `torchvision::nms` and `CUDA`:* your `torch` and `torchvision` versions do not match. Reinstall both with the command from pytorch.org, or uninstall them and run `pip install ultralytics` again to get matching CPU versions.

## Step 3.3: Test your detector

Test the app under different conditions and note, for each, whether the object is detected reliably, only sometimes or not at all:

- bright light and dim light
- a plain background and a cluttered background
- close to the camera and far away (for example 2 m)
- the object partly hidden (for example by your hand)
- a **different** object that looks similar to yours (does the model detect it by mistake?)

Also note the inference time and FPS that the app shows.

**Report:** Take a screenshot of the app detecting your object, with the inference time and FPS visible, for **Figure 5** in **Section 4.1**. Enter your test results in the table in **Section 4.2**.

# Lab report and submission

Answer **Q1–Q4 in Section 5** of the **Lab 3 Report Template** and write your reflection in **Section 6**. Written answers are required **only in the report**, not in this notebook.

**Lab demonstration:** show your working webcam app to the lab assistant, with your object. Have your laptop set up and the app ready to run. Be prepared to explain your dataset, your training results and your answers.

**Submission on Canvas:** this notebook (saved with its outputs) and your report as one PDF.

Good luck!